|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Timeline profiling<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: read the timeline<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys, sqlite3, statistics, subprocess, shutil
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
TRACES = ROOT / '.cudacache' / 'practicum_i'
TRACES.mkdir(parents=True, exist_ok=True)
print('nsys:', shutil.which('nsys') or 'NOT FOUND. It comes with the CUDA toolkit, in /usr/local/cuda/bin.')

# Read the timeline

A timeline shows two machines side by side: the CPU, which launches work, and
the GPU, which runs it. A step is as slow as the slower of the two, plus every
moment where one waits for the other. You find the slower side first. Every
optimization depends on the answer.

This notebook profiles a real decode loop of Qwen3-1.7B with **Nsight
Systems** (`nsys`). The loop marks its phases with NVTX ranges: `step`,
`forward` (the CPU time to launch one forward pass) and `cpu_work` (8 ms of
busy Python, as a scheduler and a detokenizer cost). You read the trace from
its SQLite export with plain SQL.

The loop runs in three versions:

- `none`: no CPU work.
- `series`: after each step, the loop waits for the GPU, and then it works.
  This is the loop of Ticket 3 of the incident file.
- `overlap`: the loop works while the GPU runs the step that it launched.

Nsight Systems needs no special permission. Each profile loads the model, so
each run takes about a minute, and the whole notebook about ten.

In [ ]:
DRIVER = r"""
import sys, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODE, CPU_MS, BATCH, TOKENS = sys.argv[1], float(sys.argv[2]), int(sys.argv[3]), 40
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B', dtype=torch.bfloat16).cuda().eval()
nvtx = torch.cuda.nvtx

def cpu_work():                          # the scheduler and the detokenizer: busy Python
    end = time.perf_counter() + CPU_MS / 1000
    while time.perf_counter() < end:
        pass

with torch.inference_mode():
    ids = tokenizer(['The three largest cities in Japan are'] * BATCH, return_tensors='pt').input_ids.cuda()
    out = model(ids, use_cache=True)
    cache, token = out.past_key_values, out.logits[:, -1:].argmax(-1)
    for _ in range(5):                   # warm up, outside the window of the profile
        out = model(token, past_key_values=cache, use_cache=True)
        token = out.logits[:, -1:].argmax(-1)
    torch.cuda.synchronize()
    start = time.perf_counter()
    torch.cuda.cudart().cudaProfilerStart()
    for step in range(TOKENS):
        nvtx.range_push('step')
        nvtx.range_push('forward')
        out = model(token, past_key_values=cache, use_cache=True)   # the CPU launches, the GPU runs later
        token = out.logits[:, -1:].argmax(-1)
        nvtx.range_pop()
        if MODE == 'series':
            token.tolist()               # wait for the GPU, THEN do the CPU work
        nvtx.range_push('cpu_work')
        cpu_work()
        nvtx.range_pop()
        nvtx.range_pop()
    torch.cuda.synchronize()
    torch.cuda.cudart().cudaProfilerStop()
    print('STEP_MS', (time.perf_counter() - start) / TOKENS * 1000)
"""
(TRACES / 'decode_loop.py').write_text(DRIVER)

def run(mode, cpu_ms, batch, profiled=True):
    """Run the loop. -> (the path of the SQLite trace or None, the step in ms)."""
    command = [sys.executable, str(TRACES / 'decode_loop.py'), mode, str(cpu_ms), str(batch)]
    trace = TRACES / f'{mode}_{batch}'
    if profiled:
        command = ['nsys', 'profile', '--trace=cuda,nvtx', '--capture-range=cudaProfilerApi',
                   '--capture-range-end=stop',            # let the program finish, and print
                   '--export=sqlite', '--force-overwrite=true', '-o', str(trace)] + command
    result = subprocess.run(command, capture_output=True, text=True, cwd=TRACES)
    step = [float(line.split()[1]) for line in result.stdout.splitlines() if line.startswith('STEP_MS')]
    if not step:
        raise RuntimeError(result.stdout[-1500:] + result.stderr[-1500:])
    return (Path(f'{trace}.sqlite') if profiled else None), step[0]

def query(db, sql, *arguments):
    return sqlite3.connect(db).execute(sql, arguments).fetchall()

# Exercise 1: read the kernels and the ranges

Write three functions. `merge` matters: two kernels on two streams can run at
the same time, and a sum of their durations would count that time twice.

In [ ]:
def kernel_intervals(db):
    """-> a sorted list of (start_ms, end_ms), one for each GPU kernel.
    The table is CUPTI_ACTIVITY_KIND_KERNEL. Its times are in nanoseconds."""
    ...

def merge(intervals):
    """-> the same time, with the overlaps joined. Two kernels on two streams
    can overlap, and a sum of their durations counts that time twice."""
    ...

def ranges(db, name):
    """-> the (start_ms, end_ms) of each NVTX range with this text, in order.
    The table is NVTX_EVENTS. A range that never ended has end NULL."""
    ...

db, _ = run('none', 0.0, 1)
print(len(kernel_intervals(db)), 'kernels,', len(ranges(db, 'step')), 'steps')

# Exercise 2: which side is slower?

For the loop with no CPU work, at batch 1 and at batch 256, compute the step
time, the GPU busy time of one step, the duration of the `forward` range, the
lag, and the idle fraction of the GPU.

Before you run it, predict: at batch 1, is the CPU or the GPU the slower side?
And at batch 256?

Look closely at `cpu_forward_ms` when the GPU is the slower side. It is not the
time to launch. When the GPU falls behind, the queue of launches fills, and
each new launch waits for a free place. So the CPU seems slow because it waits
for the GPU.

In [ ]:
def step_profile(db):
    """-> a dict for the window from the first 'step' to the end of the last:
         step_ms         the mean time of one step
         gpu_ms          the mean GPU busy time of one step (merged kernels)
         cpu_forward_ms  the mean duration of the 'forward' range
         lag_ms          the mean time from the end of a 'forward' range to the
                         end of the last kernel that it launched
         idle            the fraction of the window with no kernel running
    A kernel and the CPU call that launched it share a correlationId. The
    calls are in the table CUPTI_ACTIVITY_KIND_RUNTIME."""
    ...

profiles = {}
for batch in (1, 256):
    db, _ = run('none', 0.0, batch)
    profiles[batch] = step_profile(db)
    print(f'batch {batch:3d}:', {key: round(value, 2) for key, value in profiles[batch].items()})

# Exercise 3: what does 8 ms of CPU work cost?

Write `slower_side` and `predict_extra`. The second one predicts how much 8 ms
of CPU work adds to each step of the `series` loop and of the `overlap` loop,
at each batch size. Think about which side waits for which. When can the CPU
work hide behind the GPU? When the series loop waits for the GPU, what is it
still waiting for? Then the cell measures both loops at both batch sizes.

In [ ]:
CPU_MS = 8.0

def slower_side(profile):
    """-> 'CPU' or 'GPU', from the profile of the loop with no work."""
    ...

def predict_extra(profile, cpu_ms, mode):
    """The time in ms that `cpu_ms` of CPU work ADDS to one step, over the
    loop with no work.
      'series':  the loop waits for the GPU, and then it works.
      'overlap': the loop works while the GPU runs the step that it launched."""
    ...

rows = []
for batch in (1, 256):
    for mode in ('series', 'overlap'):
        db, _ = run(mode, CPU_MS, batch)
        extra = step_profile(db)['step_ms'] - profiles[batch]['step_ms']
        predicted = predict_extra(profiles[batch], CPU_MS, mode)
        rows.append((batch, mode, predicted, extra))
        print(f'batch {batch:3d} ({slower_side(profiles[batch])} slower) {mode:8s}: '
              f'predicted +{predicted:4.1f} ms, measured +{extra:4.1f} ms')

### The checks

The checks test two things. First, the machine: the facts that every run
shows. Second, your model: it must name the slower side at each batch size,
predict the series loop within 2.8 ms (35% of the CPU work), and hide more work
when the GPU is the slower side. Two profiled runs of the same loop can differ
by a few milliseconds, so the checks test the effect, not the third digit.

If your model predicts that the overlap at batch 256 hides **all** of the CPU
work, compare with the measurement, and read the `forward` range of each loop.
The difference is the most interesting number of this notebook.

In [ ]:
### THE CHECKS. Do not edit this cell.

assert len(profiles) == 2 and len(rows) == 4, 'run Exercises 2 and 3 first: a check with no data proves nothing'
assert slower_side(profiles[1]) == 'CPU' and slower_side(profiles[256]) == 'GPU', \
    'at batch 1 the CPU is the slower side here, and at batch 256 the GPU'
measured = {(batch, mode): extra for batch, mode, _, extra in rows}
predicted = {(batch, mode): value for batch, mode, value, _ in rows}
# the machine: the facts that every run shows
assert measured[(1, 'overlap')] > 0.7 * CPU_MS, 'with the CPU slower, the overlap must still pay the CPU work'
assert measured[(256, 'overlap')] < 0.6 * CPU_MS, 'with the GPU slower, the overlap must hide most of the CPU work'
for batch in (1, 256):
    assert 0.8 * CPU_MS < measured[(batch, 'series')] < 1.5 * CPU_MS, 'the series loop pays the CPU work, plus a little'
# your model: the same facts, predicted
assert predicted[(256, 'overlap')] < predicted[(1, 'overlap')], 'your model must hide more work when the GPU is slower'
for batch in (1, 256):
    assert abs(predicted[(batch, 'series')] - measured[(batch, 'series')]) < 0.35 * CPU_MS, \
        f'batch {batch} series: predicted +{predicted[(batch, "series")]:.1f} ms, measured +{measured[(batch, "series")]:.1f} ms'
print('All the checks pass: you found the slower side, and what the CPU work costs on each side.')

# Exercise 4: the profiler changes what it measures

Run the loop at batch 1 with no profiler, and compare the step time.

In [ ]:
_, plain = run('none', 0.0, 1, profiled=False)
print(f'batch 1, no profiler: {plain:.1f} ms for each step')
print(f'batch 1, under nsys : {profiles[1]["step_ms"]:.1f} ms for each step')
print(f'the profiler adds {profiles[1]["step_ms"] / plain - 1:.0%}')

# Exercise 5: your engine (after stage 28)

`./vc nsys` profiles your capstone engine and writes
`.cudacache/engine_profile.nsys-rep`. Export it to SQLite:

    nsys export --type sqlite .cudacache/engine_profile.nsys-rep

Each step of the engine is an NVTX range named `engine_step_<n>`. Use your
functions of Exercise 1, with `text LIKE 'engine_step_%'`, and answer:

1. What fraction of the window is the GPU idle between the steps?
2. Is the CPU or the GPU the slower side of your engine at the load of the
   profile? What would change the answer?
3. Stage 23 captures the decode step as a graph. What does the graph remove
   from the timeline, and which side does it speed up?

### Before you open the solution

1. In Exercise 3, why does the overlap loop still pay for the CPU work when
   the CPU is the slower side?
2. A GPU that is idle 30% of a step is a clue, not a diagnosis. Name two
   different causes that give the same idle fraction, and the evidence on the
   timeline that separates them.
3. The profiler slowed the loop down. Is a conclusion from a profiled run
   still valid? Which conclusions survive, and which do not?